# Algoritmo escolhido: KNN
Como funciona:
- Armazena todo o conjunto de treino.
- Para classificar um novo ponto, calcula-se a distância (ex: Euclidiana) para todos os   pontos de treino.
- Seleciona os k vizinhos mais próximos.
- A classe é decidida por votação maioritária.

Razões por trás da escolha deste algoritmo:
- Simplicidade de implementação e interpretação.
- Facilmente modificável (pesos, kernels, remoção de pontos).
- Sensível ao desbalanceamento 


Hipóteses:
k-NN padrão será fortemente afetado pelo desequilíbrio. Como a classificação é feita por votação maioritária dos k vizinhos mais próximos, em datasets desequilibrados, a probabilidade de um ponto da classe minoritária ter vizinhos da classe majoritária é elevada, levando a um elevado número de Falsos Negativos e um baixo Recall.

In [ ]:
#Função de knn normal

import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split # Importação permitida

class KNearstNeighbor:
    
    def __init__(self):
        # Inicializar as variáveis para podermos aceder a elas em qualquer parte da classe
        self.x_train = None
        self.y_train = None
        self.x_test = None
        self.y_test = None

    def load_split_train(self, X, y, test_size=0.2, random_state=42):
        """
        Recebe o dataset completo, faz o split de treino/teste, garante a estratificação 
        para dados desequilibrados e guarda os dados de treino (faz o fit).
        Retorna os dados de teste para poderes usar na avaliação.
        """
        # 1. Fazer o split
        self.x_train, self.x_test, self.y_train, self.y_test = train_test_split(
            X, y, 
            test_size=test_size, 
            random_state=random_state, 
            stratify=y # Mantém a proporção da classe minoritária
        )
        
        # O KNN não tem matemática de treino, apenas guarda os dados na memória.
        # Como já guardámos o x_train e y_train na linha acima, o "fit" está feito.
        
        return self.x_test, self.y_test

    # Método train original
    def train(self, x_train, y_train):
        self.x_train = x_train
        self.y_train = y_train
    
    def predict(self, x_data, k=1):
        nearst_neighbors = []
        for x in x_data:
            distances_between_points = self.__euclidean_distance(
                point_a=x,
                points_b=self.x_train,
                labels_b=self.y_train)

            # sort by distance and carry label within
            sorted_distances = sorted(
                distances_between_points, key=lambda t: t[0])
            nearst_neighbors.append(
                sorted_distances[:k]
            )

        return self.__vote(nearst_neighbors)

    def __vote(self, nearst_neighbors):
        predicts = []
        for neighbors in nearst_neighbors:
            label_counter = Counter([neighbor[1] for neighbor in neighbors])
            [(predicted_label, _)] = label_counter.most_common(1)
            predicts.append([predicted_label])
        return predicts

    def __euclidean_distance(self, point_a, points_b, labels_b):
        distances = []
        size_points_b = len(points_b)
        for i in range(size_points_b):
            point = points_b[i]
            #garante que lê bem a label seja array 1D ou 2D
            point_label = labels_b[i][0] if isinstance(labels_b[i], (list, np.ndarray)) else labels_b[i]

            distance = np.sqrt(np.sum(np.square(
                    point_a - point)))
            distances.append(
                (distance, point_label))
        return distances

    @staticmethod # Transformado em staticmethod para o poderes chamar facilmente sem precisar do 'self'
    def avaliar_desequilibrio(y_real, y_pred, classe_minoritaria):
        y_real = np.array(y_real).flatten()
        y_pred = np.array(y_pred).flatten()
        
        # Verdadeiros Positivos, Falsos Positivos, etc.
        TP = np.sum((y_pred == classe_minoritaria) & (y_real == classe_minoritaria))
        FP = np.sum((y_pred == classe_minoritaria) & (y_real != classe_minoritaria))
        FN = np.sum((y_pred != classe_minoritaria) & (y_real == classe_minoritaria))
        
        # Calcular métricas (com proteção contra divisão por zero)
        precision = TP / (TP + FP) if (TP + FP) > 0 else 0
        recall = TP / (TP + FN) if (TP + FN) > 0 else 0
        f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        accuracy = np.mean(y_pred == y_real)
        
        print(f"Accuracy:  {accuracy:.4f}")#(Enganadora em dados desequilibrados)
        print(f"Precision: {precision:.4f}")#(Quando diz que é anomalia, qual a % de acerto?)
        print(f"Recall:    {recall:.4f}")#(Das anomalias totais, quantas conseguiu apanhar?)
        print(f"F1-Score:  {f1_score:.4f}")#(O equilíbrio entre Precision e Recall)
        
        return accuracy, precision, recall, f1_score

In [3]:
import os
import pandas as pd
import numpy as np
from collections import Counter

# Pasta com os ficheiros CSV
pasta_datasets = 'class_imbalance'

# Instanciar o modelo
knn_normal = KNearstNeighbor()

# Iniciar o ciclo for para percorrer a pasta com os datasets
for ficheiro in os.listdir(pasta_datasets):
    
    # Garantir que só lê ficheiros de dados (ex: .csv)
    if ficheiro.endswith('.csv'):
        print("-" * 40 + "\n")
        print(f" A testar o dataset: {ficheiro}")
        
        caminho_completo = os.path.join(pasta_datasets, ficheiro)
        
        
        df = pd.read_csv(caminho_completo)
        
        # Separar features (X) da target (y)
        X_df = df.iloc[:, :-1]
        y_completo = df.iloc[:, -1].values
        
        # Transformar todas as colunas de TEXTO em NÚMEROS (One-Hot Encoding)
        # O drop_first=True evita colinearidade matemática perfeita
        X_df = pd.get_dummies(X_df, drop_first=True)
        
        # Preencher possíveis valores vazios (NaN) com 0 
        X_df = X_df.fillna(0)
        
        # Converter para array do NumPy e garantir que é tudo número (float)
        X_completo = X_df.values.astype(float)
        
        # DETETAR A CLASSE MINORITÁRIA AUTOMATICAMENTE
        contagem = Counter(y_completo)
        classe_minoritaria = min(contagem, key=contagem.get)
        
        print(f"Distribuição total do dataset: {dict(contagem)}")
        print(f"A Classe Minoritária é: '{classe_minoritaria}'\n")
        
        # SPLIT + FIT
        x_test, y_test = knn_normal.load_split_train(X_completo, y_completo, test_size=0.3)
        
        # PREVER
        previsoes = knn_normal.predict(x_test, k=5)
        
        # AVALIAR
        print("--- Resultados KNN ---")
        KNearstNeighbor.avaliar_desequilibrio(y_test, previsoes, classe_minoritaria=classe_minoritaria)
        print("\n\n")

----------------------------------------

 A testar o dataset: dataset_978_mfeat-factors.csv
Distribuição total do dataset: {'P': 200, 'N': 1800}
A Classe Minoritária é: 'P'

--- Resultados KNN ---
Accuracy:  0.9883
Precision: 0.9649
Recall:    0.9167
F1-Score:  0.9402



----------------------------------------

 A testar o dataset: dataset_947_arsenic-male-bladder.csv
Distribuição total do dataset: {'N': 24, 'P': 535}
A Classe Minoritária é: 'N'

--- Resultados KNN ---
Accuracy:  0.9762
Precision: 1.0000
Recall:    0.4286
F1-Score:  0.6000



----------------------------------------

 A testar o dataset: dataset_1004_synthetic_control.csv
Distribuição total do dataset: {'N': 500, 'P': 100}
A Classe Minoritária é: 'P'

--- Resultados KNN ---
Accuracy:  1.0000
Precision: 1.0000
Recall:    1.0000
F1-Score:  1.0000



----------------------------------------

 A testar o dataset: dataset_1056_mc1.csv
Distribuição total do dataset: {np.False_: 9398, np.True_: 68}
A Classe Minoritária é: '

# Algoritmo KNN com várias alteraões:
- atribuímos maior peso a classes e atributos
  (O __update_internal_weights calcula a correlação entre cada característica e classe alvo senod que características que tÊm pouca correlação com a classe alvo recebem pesos menores)
- RBF Kernel (o método __rbf_similarity transforma a distÂncia numa pontuação de similaridade entre 0 e 1 sendo que o parâmetro gama controla o raio de influÊncia - vizinhos próximos ganham muito mais importância)
- implementámos o sequential selection, começando com um ponto de cada classe para não não demorar muito a encontrar o primeiro erro caso o primeiro nó fosse da classe maioritária


In [4]:
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split

class CW_RBF_KNN:
    def __init__(self, gamma=0.1):
        # Hiperparâmetro que controla a largura da "bolha" de influência do Kernel RBF
        self.gamma = gamma
        self.class_weights = {}
        self.feature_weights = []
        self.x_train = None
        self.y_train = None

    def load_split_train(self, X, y, test_size=0.3, random_state=42):
        """
        Recebe o dataset completo, faz o split de treino/teste, garante a estratificação 
        para dados desequilibrados e inicia o processo de Insertion Selection.
        """
        x_train_raw, x_test, y_train_raw, y_test = train_test_split(
            X, y, 
            test_size=test_size, 
            random_state=random_state, 
            stratify=y # Mantém a proporção da classe minoritária
        )
        
        # O método train aqui executa a Sequential Insertion Selection (CNN)
        self.train(x_train_raw, y_train_raw)
        
        return x_test, y_test

    def train(self, x_raw, y_raw):
        # Converter para arrays numpy e achatar y_raw
        x_raw = np.array(x_raw)
        y_raw = np.array(y_raw).flatten()
        
        # 1. Inicialização: Pegar o primeiro exemplar de cada classe para começar
        unique_labels = np.unique(y_raw)
        initial_indices = [np.where(y_raw == lbl)[0][0] for lbl in unique_labels]
        
        self.x_train = x_raw[initial_indices]
        self.y_train = y_raw[initial_indices]
        
        # Cálculo inicial de pesos (necessário para o predict funcionar durante o treino)
        self.__update_internal_weights(x_raw, y_raw)

        # 2. Sequential Insertion Selection (Loop de Condensação)
        changed = True
        while changed:
            changed = False
            # Percorrer os dados originais
            for i in range(len(x_raw)):
                # Tentar prever com o que temos atualmente no self.x_train
                # Usamos k=1 para a lógica clássica do Condensed NN
                pred = self.predict([x_raw[i]], k=1)[0]
                
                # Se o modelo ERROU, o ponto i é importante para definir a fronteira
                if pred != y_raw[i]:
                    # vstack mantém a estrutura de matriz
                    self.x_train = np.vstack([self.x_train, x_raw[i]])
                    self.y_train = np.append(self.y_train, y_raw[i])
                    
                    # Recalcular pesos internos com o novo ponto incluído
                    self.__update_internal_weights(self.x_train, self.y_train)
                    changed = True 

    def __update_internal_weights(self, x_data, y_data):
        """Função auxiliar para atualizar pesos de classes e atributos durante a inserção"""
        total_samples = len(y_data)
        counts = Counter(y_data)
        
        # PESOS DAS CLASSES (Suavizados com Raiz Quadrada)
        # Dá mais poder de voto à classe minoritária
        self.class_weights = {label: np.sqrt(total_samples / count) for label, count in counts.items()}

        # PESOS DAS VARIÁVEIS (Correlação Absoluta)
        if len(counts) > 1:
            unique_labels = list(counts.keys())
            y_numeric = np.where(y_data == unique_labels[0], 0, 1)
            
            f_weights = []
            for i in range(x_data.shape[1]):
                coluna = x_data[:, i]
                if np.std(coluna) == 0:
                    f_weights.append(0.0)
                else:
                    corr = np.corrcoef(coluna, y_numeric)[0, 1]
                    f_weights.append(np.abs(corr) if not np.isnan(corr) else 0.0)
            self.feature_weights = np.array(f_weights)
        else:
            self.feature_weights = np.ones(x_data.shape[1])

    def predict(self, x_data, k=1):
        x_data = np.array(x_data)
        predicts = []
        for x in x_data:
            # Calcular a Similaridade RBF em relação a todos os pontos de treino selecionados
            similarities = self.__rbf_similarity(x)
            # Ordenar de forma DECRESCENTE (no RBF, maior = melhor/mais perto)
            sorted_similarities = sorted(similarities, key=lambda t: t[0], reverse=True)
            k_nearest = sorted_similarities[:k]
            predicts.append(self.__vote(k_nearest))
        return np.array(predicts)

    def __rbf_similarity(self, point_a):
        # Diferença ponderada pelos feature_weights (Correlação)
        diferenca_quadrado = np.square(self.x_train - point_a)
        distancia_ponderada = np.sum(self.feature_weights * diferenca_quadrado, axis=1)
        # Aplicar o Kernel RBF
        similarities = np.exp(-self.gamma * distancia_ponderada)
        return list(zip(similarities, self.y_train))

    def __vote(self, neighbors):
        class_scores = {}
        for similarity, label in neighbors:
            # Voto = Peso da Classe * Similaridade RBF
            vote_power = self.class_weights.get(label, 1.0) * similarity
            class_scores[label] = class_scores.get(label, 0) + vote_power
        # Devolve a classe com a maior pontuação
        return max(class_scores, key=class_scores.get)

    def evaluate(self, x_test, y_test, k=1):
        """Calcula métricas focadas em desequilíbrio para a classe minoritária."""
        y_real = np.array(y_test).flatten()
        y_pred = self.predict(x_test, k=k)
        
        counts = Counter(y_real)
        minority = min(counts, key=counts.get)
        
        tp = np.sum((y_pred == minority) & (y_real == minority))
        fp = np.sum((y_pred == minority) & (y_real != minority))
        fn = np.sum((y_pred != minority) & (y_real == minority))
        
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * (prec * rec) / (prec + rec) if (prec + rec) > 0 else 0
        acc = np.mean(y_pred == y_real)
        
        return acc, prec, rec, f1

In [5]:
import os

# Pasta com os ficheiros CSV
pasta_datasets = 'class_imbalance'

# Cabeçalho da Tabela para a avaliação empírica
header = f"{'Dataset':<35} | {'Orig':>6} | {'Sel':>6} | {'Acc':>7} | {'Prec':>6} | {'Rec':>6} | {'F1':>6}"
print(header)
print("-" * len(header))

for ficheiro in os.listdir(pasta_datasets):
    if ficheiro.endswith('.csv'):
        try:
            caminho_completo = os.path.join(pasta_datasets, ficheiro)
            df = pd.read_csv(caminho_completo)
            
            # Separar features (X) da target (y)
            X_df = df.iloc[:, :-1]
            y_completo = df.iloc[:, -1].values
            
            # Transformar todas as colunas de TEXTO em NÚMEROS (One-Hot Encoding)
            # O drop_first=True evita colinearidade matemática perfeita
            X_df = pd.get_dummies(X_df, drop_first=True)
            
            # Preencher possíveis valores vazios (NaN) com 0 
            X_df = X_df.fillna(0)
            
            # Converter para array do NumPy e garantir que é tudo número (float)
            X_completo = X_df.values.astype(float)
            
            # Instanciar o modelo (O gamma controla a sensibilidade do RBF)
            model = CW_RBF_KNN(gamma=0.1)
            
            # Calcular tamanho do treino (antes da compressão) para a tabela
            orig_train_size = int(len(y_completo) * 0.7) 
            
            # SPLIT + FIT (A seleção por inserção acontece aqui dentro)
            x_test, y_test = model.load_split_train(X_completo, y_completo, test_size=0.3)
            
            # PREVER E AVALIAR
            acc, prec, rec, f1 = model.evaluate(x_test, y_test, k=5)
            
            # Tamanho após a Selection (pontos que ficaram na memória)
            sel_size = len(model.x_train)
            
            # Imprimir linha formatada
            print(f"{ficheiro[:35]:<35} | {orig_train_size:>6} | {sel_size:>6} | {acc*100:>6.1f}% | {prec:>6.2f} | {rec:>6.2f} | {f1:>6.2f}")
            
        except Exception as e:
            print(f"Erro no dataset {ficheiro}: {e}")

print("-" * len(header))
print("Avaliação empírica concluída.")

Dataset                             |   Orig |    Sel |     Acc |   Prec |    Rec |     F1
------------------------------------------------------------------------------------------
dataset_978_mfeat-factors.csv       |   1400 |    115 |   94.7% |   0.97 |   0.48 |   0.64
dataset_947_arsenic-male-bladder.cs |    391 |     75 |   91.7% |   0.23 |   0.43 |   0.30
dataset_1004_synthetic_control.csv  |    420 |      8 |  100.0% |   1.00 |   1.00 |   1.00
Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "/opt/anaconda3/envs/ac1/lib/python3.11/site-packages/IPython/core/interactiveshell.py", line 3699, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/var/folders/ws/v_d_g0rn7178ft7w39nl11m00000gn/T/ipykernel_82726/3355816082.py", line 38, in <module>
    x_test, y_test = model.load_split_train(X_completo, y_completo, test_size=0.3)
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/ws/v_d_g0rn7178ft7w39nl11m00000gn/T/ipykernel_82726/422528883.py", line 28, in load_split_train
    self.train(x_train_raw, y_train_raw)
  File "/var/folders/ws/v_d_g0rn7178ft7w39nl11m00000gn/T/ipykernel_82726/422528883.py", line None, in train
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/opt/anaconda3/envs/ac1/lib/python3.11/site-packages/IPython/core/interactiveshell.py", line 2194, in showtraceback
   

Problemas com este código: acaba por demorar mais tempo (possivelmente às previsões mais complexas e ao ter que percorrer o while do insertion) - deverá haver um ficheiro com muito ruído (acho que é o dataset_38_sick)